# Question 2

In [85]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from pathlib import Path
from scipy.stats import chi2 as scipy_chi2
 
from nfw_theory import NFWHalo_theory   # provided in the assessment
 
 
# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR   = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
 
HALO_POS_FILE   = DATA_DIR / "halo_positions.txt"
SOURCE_POS_FILE = DATA_DIR / "source_positions.txt"
 
# Q1 ellipticity outputs
ELLIP_FILES = {
    "unweighted": OUTPUT_DIR / "ellipticities_unweighted.npy",
    "weighted":   OUTPUT_DIR / "ellipticities_weighted.npy",
    "hsm":        OUTPUT_DIR / "ellipticities_hsm.npy",
}
 
# Simulation truth parameters (fixed throughout Q2)
HALO_REDSHIFT   = 0.3
CONCENTRATION   = 4.0
OMEGA_M         = 0.3
OMEGA_LAM       = 0.7
 
# Radial bins: 10 log-spaced bins in [100, 3600] arcsec
N_BINS      = 10
BIN_EDGES   = np.logspace(np.log10(100.0), np.log10(3600.0), N_BINS + 1)
BIN_CENTRES = np.sqrt(BIN_EDGES[:-1] * BIN_EDGES[1:])   # geometric mean

COLOURS = {"unweighted": "steelblue", "weighted": "tomato", "hsm": "seagreen"}
LABELS  = {
    "unweighted": "Unweighted moments",
    "weighted":   "Weighted moments (FWHM=4px)",
    "hsm":        "HSM",
}

In [86]:
def load_positions(path: Path) -> np.ndarray:
    """Load (x, y) positions in arcseconds from a two-column text file."""
    pos = np.loadtxt(path)          # shape (N, 2)
    return pos                      # columns: x [arcsec], y [arcsec]

In [87]:
def compute_tangential_ellipticities(
    halo_pos: np.ndarray,
    source_pos: np.ndarray,
    e1: np.ndarray,
    e2: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute per-pair tangential ellipticity and separation.
 
    For each (halo h, source s) pair:
        dx    = x_s - x_h
        dy    = y_s - y_h
        r     = sqrt(dx^2 + dy^2)          [arcsec]
        phi   = arctan2(dy, dx)
        e_t   = -(e1 * cos(2*phi) + e2 * sin(2*phi))
 
    Sources with NaN ellipticity are excluded (flagged with NaN e_t).
 
    Parameters
    ----------
    halo_pos   : (N_h, 2) array of halo (x, y) positions [arcsec]
    source_pos : (N_s, 2) array of source (x, y) positions [arcsec]
    e1, e2     : (N_s,) arrays of source ellipticity components
 
    Returns
    -------
    r_pairs  : (N_h * N_s,) separations [arcsec]
    et_pairs : (N_h * N_s,) tangential ellipticities (NaN for failed sources)
    """
    n_h = len(halo_pos)
    n_s = len(source_pos)
 
    # Broadcast to (N_h, N_s)
    dx  = source_pos[:, 0][np.newaxis, :] - halo_pos[:, 0][:, np.newaxis]
    dy  = source_pos[:, 1][np.newaxis, :] - halo_pos[:, 1][:, np.newaxis]
    r   = np.sqrt(dx**2 + dy**2)     # (N_h, N_s)
    phi = np.arctan2(dy, dx)          # (N_h, N_s)
 
    # Broadcast ellipticities (N_s,) -> (N_h, N_s)
    e1_2d = e1[np.newaxis, :]
    e2_2d = e2[np.newaxis, :]
 
    et = -(e1_2d * np.cos(2 * phi) + e2_2d * np.sin(2 * phi))   # (N_h, N_s)
 
    return r.ravel(), et.ravel()

In [88]:
def mean_tangential_shear(
    r_pairs: np.ndarray,
    et_pairs: np.ndarray,
    bin_edges: np.ndarray,
    weights: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Bin tangential ellipticities and return mean tangential shear.
 
    The mean tangential shear is <e_t> / 2 because in the weak-lensing
    limit <e_t> = 2 * gamma_t (lectures convention).
 
    Uncertainty is estimated from the sample variance of per-pair e_t values
    within each bin (shape noise dominated):
        Var(gamma_t_bin) = Var(e_t_pairs) / (4 * N_pairs)
 
    If weights are provided (optimal estimator), a weighted mean and the
    corresponding weighted sample variance are used instead.
 
    Parameters
    ----------
    r_pairs   : separations for all pairs [arcsec]
    et_pairs  : tangential ellipticities for all pairs
    bin_edges : radial bin edges [arcsec]
    weights   : optional per-pair inverse-variance weights
 
    Returns
    -------
    gt      : mean tangential shear per bin  (N_bins,)
    gt_err  : 1-sigma uncertainty per bin    (N_bins,)
    n_pairs : number of valid pairs per bin  (N_bins,)
    """
    n_bins  = len(bin_edges) - 1
    gt      = np.zeros(n_bins)
    gt_err  = np.zeros(n_bins)
    n_pairs = np.zeros(n_bins, dtype=int)
 
    for k in range(n_bins):
        in_bin  = (r_pairs >= bin_edges[k]) & (r_pairs < bin_edges[k + 1])
        et_bin  = et_pairs[in_bin]
        # Drop NaN (failed Q1 measurements)
        finite  = np.isfinite(et_bin)
        et_bin  = et_bin[finite]
        n       = len(et_bin)
        n_pairs[k] = n
 
        if n < 2:
            gt[k]     = np.nan
            gt_err[k] = np.nan
            continue
 
        if weights is None:
            # Simple (unweighted) mean
            mean_et      = et_bin.mean()
            var_et_pairs = et_bin.var(ddof=1)       # sample variance
        else:
            w_bin   = weights[in_bin][finite]
            w_sum   = w_bin.sum()
            mean_et = (w_bin * et_bin).sum() / w_sum
            # Weighted sample variance of et values
            var_et_pairs = (w_bin * (et_bin - mean_et)**2).sum() / w_sum
 
        gt[k]     = mean_et / 2.0
        # Var(mean_et) = Var(et_pairs) / N  ->  Var(gt) = Var(et_pairs) / (4*N)
        gt_err[k] = np.sqrt(var_et_pairs / (4.0 * n))
 
    return gt, gt_err, n_pairs

In [89]:
def estimate_shape_noise_variance(et_pairs: np.ndarray) -> float:
    """
    Estimate the per-galaxy shape-noise variance from all valid tangential
    ellipticity measurements.
 
    The shape noise dominates the per-pair scatter (the shear signal is tiny
    compared to intrinsic ellipticities), so Var(e_t) ~ sigma_e^2.
 
    Returns sigma_e^2 per component.
    """
    finite = np.isfinite(et_pairs)
    return float(np.var(et_pairs[finite], ddof=1))

In [90]:
def optimal_weights(et_pairs: np.ndarray) -> np.ndarray:
    """
    Return per-pair inverse-variance weights  w_i = 1 / sigma_e^2.
 
    Since the shape noise is the same for all galaxies in this simulation,
    the weights are uniform and the optimal estimator reduces to the simple
    mean.  In real surveys galaxies have different sizes and S/N, so
    sigma_e^2 would vary per source and the weights would differ.
 
    NaN pairs receive weight 0.
    """
    sigma2 = estimate_shape_noise_variance(et_pairs)
    w      = np.where(np.isfinite(et_pairs), 1.0 / sigma2, 0.0)
    return w

In [91]:
def plot_tangential_shear_profiles(
    profiles: dict,
    output_dir: Path,
    filename: str = "q2c_tangential_shear_profiles.png",
):
    """
    Plot gamma_t(r) with 1-sigma error bars for all three Q1 methods,
    both simple and optimal estimators, on a log-log scale.
 
    Parameters
    ----------
    profiles : dict keyed by method name, each value a dict with keys
               'gt_simple', 'gt_err_simple', 'gt_opt', 'gt_err_opt'
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
 
    for ax, estimator in zip(axes, ["simple", "optimal"]):
        for method in ["unweighted", "weighted", "hsm"]:
            gt     = profiles[method][f"gt_{estimator}"]
            gt_err = profiles[method][f"gt_err_{estimator}"]
            valid  = np.isfinite(gt)
            ax.errorbar(
                BIN_CENTRES[valid], gt[valid], yerr=gt_err[valid],
                fmt="o", ms=5, capsize=3,
                color=COLOURS[method], label=LABELS[method],
            )
 
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(r"Separation $\theta$ [arcsec]", fontsize=13)
        ax.set_title(
            f"{'Simple (unweighted) mean' if estimator == 'simple' else 'Optimal (inverse-variance) mean'}",
            fontsize=11,
        )
        ax.legend(fontsize=9)
        ax.grid(True, which="both", ls=":", alpha=0.4)
 
    axes[0].set_ylabel(r"Mean tangential shear $\langle\gamma_t\rangle$", fontsize=13)
    fig.suptitle(
        "Mean tangential shear profiles around NFW halos\n"
        "Three shape estimators compared",
        fontsize=13,
    )
    fig.tight_layout()
    out = output_dir / filename
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved {out}")

In [92]:
def build_nfw_theory(mass: float, source_redshift: float) -> np.ndarray:
    """
    Compute the theoretical mean tangential shear at BIN_CENTRES for a
    single NFW halo with the simulation truth concentration / cosmology.
 
    Parameters
    ----------
    mass            : halo mass [M_sun / h]  (or M_sun depending on NFWtheory)
    source_redshift : source galaxy redshift
 
    Returns
    -------
    gt_theory : (N_bins,) array of predicted mean tangential shear
    """
    nfw = NFWHalo_theory(
        mass      = mass,
        conc      = CONCENTRATION,
        redshift  = HALO_REDSHIFT,
        omega_m   = OMEGA_M,
        omega_lam = OMEGA_LAM,
    )
    return nfw.get_tangential_shear(source_redshift, BIN_CENTRES)

In [93]:
def log_likelihood(
    gt_data: np.ndarray,
    gt_err: np.ndarray,
    mass: float,
    source_redshift: float,
) -> float:
    """
    Gaussian log-likelihood:
        ln L = -0.5 * sum_k [ (gt_data_k - gt_theory_k)^2 / gt_err_k^2 ]
 
    Bins with NaN data or theory are excluded.
 
    Parameters
    ----------
    gt_data         : measured mean tangential shear per bin
    gt_err          : 1-sigma uncertainty per bin
    mass            : halo mass
    source_redshift : source redshift
 
    Returns
    -------
    lnL : float
    """
    gt_theory = build_nfw_theory(mass, source_redshift)
    valid     = np.isfinite(gt_data) & np.isfinite(gt_err) & np.isfinite(gt_theory)
    if valid.sum() == 0:
        return -np.inf
    residuals = gt_data[valid] - gt_theory[valid]
    return -0.5 * np.sum((residuals / gt_err[valid])**2)

In [94]:
def compute_posterior_grid(
    gt_data: np.ndarray,
    gt_err: np.ndarray,
    mass_grid: np.ndarray,
    zs_grid: np.ndarray,
) -> np.ndarray:
    """
    Evaluate the (unnormalised) posterior on a 2D grid of (mass, z_source)
    using a flat (uniform) prior on both parameters.
 
    Parameters
    ----------
    gt_data    : (N_bins,) measured tangential shear
    gt_err     : (N_bins,) 1-sigma uncertainties
    mass_grid  : (N_m,) halo mass values to evaluate
    zs_grid    : (N_z,) source redshift values to evaluate
 
    Returns
    -------
    log_post : (N_m, N_z) array of log-posterior values
    """
    n_m = len(mass_grid)
    n_z = len(zs_grid)
    log_post = np.full((n_m, n_z), -np.inf)
 
    for i, mass in enumerate(mass_grid):
        for j, zs in enumerate(zs_grid):
            log_post[i, j] = log_likelihood(gt_data, gt_err, mass, zs)
 
    return log_post

In [95]:
def plot_joint_posterior(
    log_post: np.ndarray,
    mass_grid: np.ndarray,
    zs_grid: np.ndarray,
    output_dir: Path,
    filename: str = "q2d_joint_posterior.png",
):
    """
    Plot the 2D joint posterior of (halo mass, source redshift) as filled
    contours at the 68% and 95% credible levels, plus the MAP estimate.
 
    The credible contours are determined from the chi^2 distribution:
    for 2 parameters, Delta(chi^2) = 2.30 (68%) and 6.17 (95%).
    """
    # Convert to unnormalised posterior (subtract max for numerical stability)
    log_post_shifted = log_post - log_post.max()
    posterior        = np.exp(log_post_shifted)
 
    # MAP estimate
    idx_map  = np.unravel_index(np.argmax(log_post), log_post.shape)
    mass_map = mass_grid[idx_map[0]]
    zs_map   = zs_grid[idx_map[1]]
    print(f"  MAP: mass = {mass_map:.3e} M_sun,  z_s = {zs_map:.3f}")
 
    # Credible contour levels in 2D (Delta chi^2 = 2.30, 6.17 for 68%, 95%)
    # log_post = -chi^2/2  ->  Delta log_post = -Delta chi^2 / 2
    delta_chi2_68 = scipy_chi2.ppf(0.68, df=2)
    delta_chi2_95 = scipy_chi2.ppf(0.95, df=2)
    level_68 = np.exp(-delta_chi2_68 / 2.0)
    level_95 = np.exp(-delta_chi2_95 / 2.0)
 
    fig, ax = plt.subplots(figsize=(8, 6))
    cf = ax.contourf(
        zs_grid, mass_grid / 1e14, posterior,
        levels=[level_95, level_68, 1.0],
        colors=["#aec6e8", "#4a90d9", "#1a3f6f"],
        alpha=0.85,
    )
    ax.contour(
        zs_grid, mass_grid / 1e14, posterior,
        levels=[level_95, level_68],
        colors=["#2166ac", "#053061"],
        linewidths=1.2,
    )
    ax.axvline(zs_map,           color="k",       ls="--", lw=1.0, label=f"MAP $z_s$ = {zs_map:.2f}")
    ax.axhline(mass_map / 1e14,  color="k",       ls=":",  lw=1.0, label=f"MAP $M$ = {mass_map:.2e} $M_\\odot$")
    ax.axvline(0.6,              color="firebrick", ls="-.", lw=1.2, label="True $z_s$ = 0.6")
 
    # Proxy handles for the filled contours
    from matplotlib.patches import Patch
    handles = [
        Patch(facecolor="#4a90d9", label="68% credible region"),
        Patch(facecolor="#aec6e8", label="95% credible region"),
    ]
    leg1 = ax.legend(handles=handles, loc="upper left",  fontsize=9)
    ax.add_artist(leg1)
    ax.legend(loc="upper right", fontsize=9)
 
    ax.set_xlabel(r"Source redshift $z_s$",                      fontsize=13)
    ax.set_ylabel(r"Halo mass $M$ [$10^{14}\,M_\odot$]",         fontsize=13)
    ax.set_title("Joint posterior: halo mass and source redshift\n(HSM estimator, flat prior)", fontsize=12)
    # Zoom axes to the region where posterior has support
    ax.set_xlim(zs_grid.min(), zs_grid.max())
    ax.set_ylim(mass_grid.min() / 1e14, mass_grid.max() / 1e14)
    fig.tight_layout()
    out = output_dir / filename
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved {out}")
 
    return mass_map, zs_map

In [96]:
def plot_mass_posterior_fixed_zs(
    log_post: np.ndarray,
    mass_grid: np.ndarray,
    zs_grid: np.ndarray,
    fixed_zs: float,
    output_dir: Path,
    filename: str = "q2d_mass_posterior_fixed_zs.png",
):
    """
    Plot the marginal posterior on halo mass with source redshift fixed
    at fixed_zs, by slicing the 2D posterior at the nearest z_s grid point.
    """
    j_fixed    = np.argmin(np.abs(zs_grid - fixed_zs))
    zs_actual  = zs_grid[j_fixed]
    log_p_mass = log_post[:, j_fixed]
 
    # Normalise
    log_p_mass -= log_p_mass.max()
    p_mass      = np.exp(log_p_mass)
    # Trapezoidal normalisation
    norm   = np.trapezoid(p_mass, mass_grid)
    p_mass /= norm
 
    # Cumulative distribution via trapezoidal integration
    cdf      = np.array([np.trapezoid(p_mass[:k+1], mass_grid[:k+1]) for k in range(len(mass_grid))])
    cdf     /= cdf[-1]
    mass_lo  = mass_grid[np.searchsorted(cdf, 0.16)]
    mass_hi  = mass_grid[np.searchsorted(cdf, 0.84)]
    mass_map = mass_grid[np.argmax(p_mass)]
 
    print(f"  Fixed z_s = {zs_actual:.3f}:")
    print(f"    MAP mass = {mass_map:.3e} M_sun")
    print(f"    68% CI   = [{mass_lo:.3e}, {mass_hi:.3e}] M_sun")
 
    # Zoom window: 5-sigma either side of MAP in log-mass space
    log_map  = np.log10(mass_map)
    log_lo   = np.log10(max(mass_lo * 0.5, mass_grid[0]))
    log_hi   = np.log10(min(mass_hi * 2.0, mass_grid[-1]))
    x_lo     = 10**log_lo / 1e14
    x_hi     = 10**log_hi / 1e14
 
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(mass_grid / 1e14, p_mass * 1e14, color="steelblue", lw=2)
    ax.fill_between(
        mass_grid / 1e14, p_mass * 1e14,
        where=(mass_grid >= mass_lo) & (mass_grid <= mass_hi),
        color="steelblue", alpha=0.35, label="68% credible interval",
    )
    ax.axvline(mass_map / 1e14, color="k", ls="--", lw=1.2,
               label=f"MAP = {mass_map / 1e14:.2f} $\\times 10^{{14}}\\,M_\\odot$")
    ax.axvline(mass_lo / 1e14,  color="steelblue", ls=":", lw=1.0,
               label=f"68% CI: [{mass_lo/1e14:.2f}, {mass_hi/1e14:.2f}] $\\times10^{{14}}\\,M_\\odot$")
    ax.axvline(mass_hi / 1e14,  color="steelblue", ls=":", lw=1.0)
    ax.set_xlim(x_lo, x_hi)
    ax.set_xlabel(r"Halo mass $M$ [$10^{14}\,M_\odot$]", fontsize=13)
    ax.set_ylabel(r"Posterior $p(M \mid z_s = %.1f)$" % zs_actual, fontsize=12)
    ax.set_title(f"Mass posterior with source redshift fixed at $z_s = {zs_actual:.1f}$", fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, ls=":", alpha=0.4)
    fig.tight_layout()
    out = output_dir / filename
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved {out}")
 
    return mass_map, mass_lo, mass_hi

In [97]:
def plot_best_fit_overlay(
    gt_data: np.ndarray,
    gt_err: np.ndarray,
    mass_map: float,
    zs_map: float,
    mass_map_zs06: float,
    output_dir: Path,
    filename: str = "q2d_best_fit_overlay.png",
):
    """
    Overplot the MAP NFW theory curve on the measured tangential shear profile.
    Shows both the joint MAP (mass_map, zs_map) and the best-fit mass at
    fixed z_s = 0.6.
    """
    gt_theory      = build_nfw_theory(mass_map, zs_map)
    gt_theory_zs06 = build_nfw_theory(mass_map_zs06, 0.6)
 
    valid = np.isfinite(gt_data) & np.isfinite(gt_err)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(
        BIN_CENTRES[valid], gt_data[valid], yerr=gt_err[valid],
        fmt="o", ms=6, capsize=3, color="seagreen", zorder=5, label="HSM data",
    )
    ax.plot(BIN_CENTRES, gt_theory, color="k", lw=2,
            label=f"Joint MAP: $M={mass_map/1e14:.2f}\\times10^{{14}}\\,M_\\odot$,"
                  f" $z_s={zs_map:.2f}$")
    ax.plot(BIN_CENTRES, gt_theory_zs06, color="firebrick", lw=1.5, ls="--",
            label=f"Best fit ($z_s=0.6$): $M={mass_map_zs06/1e14:.2f}\\times10^{{14}}\\,M_\\odot$")
 
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"Separation $\theta$ [arcsec]", fontsize=13)
    ax.set_ylabel(r"Mean tangential shear $\langle\gamma_t\rangle$", fontsize=13)
    ax.set_title("MAP NFW theory vs measured tangential shear (HSM)", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, which="both", ls=":", alpha=0.4)
    fig.tight_layout()
    out = output_dir / filename
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved {out}")

In [98]:
def main():
    # ------------------------------------------------------------------
    # Load positions and Q1 ellipticities
    # ------------------------------------------------------------------
    halo_pos   = load_positions(HALO_POS_FILE)
    source_pos = load_positions(SOURCE_POS_FILE)
    print(f"Loaded {len(halo_pos)} halos and {len(source_pos)} sources")
 
    ellipticities = {
        name: np.load(path) for name, path in ELLIP_FILES.items()
    }
 
    # ------------------------------------------------------------------
    # (a) + (b)  Tangential shear profiles for all three methods
    # ------------------------------------------------------------------
    profiles = {}
    for method, e_arr in ellipticities.items():
        e1 = e_arr[:, 0]
        e2 = e_arr[:, 1]
 
        print(f"\nMethod: {method}")
        r_pairs, et_pairs = compute_tangential_ellipticities(
            halo_pos, source_pos, e1, e2
        )
 
        # (a) Simple (unweighted) mean
        gt_s, gt_err_s, n_s = mean_tangential_shear(
            r_pairs, et_pairs, BIN_EDGES, weights=None
        )
 
        # (b) Optimal (inverse-variance) weighted mean
        w = optimal_weights(et_pairs)
        gt_o, gt_err_o, n_o = mean_tangential_shear(
            r_pairs, et_pairs, BIN_EDGES, weights=w
        )
 
        sigma2 = estimate_shape_noise_variance(et_pairs)
        print(f"  Shape noise variance sigma_e^2 = {sigma2:.5f}")
        print(f"  Bin pair counts (simple): {n_s}")
 
        profiles[method] = {
            "gt_simple":    gt_s,
            "gt_err_simple": gt_err_s,
            "n_simple":     n_s,
            "gt_optimal":   gt_o,
            "gt_err_optimal": gt_err_o,
            "n_optimal":    n_o,
            # Aliases used by plotting helpers
            "gt_opt":       gt_o,
            "gt_err_opt":   gt_err_o,
        }
 
    # Save profiles for inspection
    np.save(OUTPUT_DIR / "tangential_shear_profiles.npy", profiles)
 
    # ------------------------------------------------------------------
    # (c) Plot: all three methods side by side
    # ------------------------------------------------------------------
    print("\nPlotting tangential shear profiles...")
    plot_tangential_shear_profiles(profiles, OUTPUT_DIR)
 
    # ------------------------------------------------------------------
    # (d) Posterior inference using HSM (most reliable estimator)
    # ------------------------------------------------------------------
    print("\nRunning posterior grid for HSM estimator...")
    gt_hsm     = profiles["hsm"]["gt_simple"]
    gt_err_hsm = profiles["hsm"]["gt_err_simple"]
 
    # Grid over mass and source redshift
    # Mass: 10^13.5 to 10^15.5 M_sun, log-spaced
    # z_s : 0.2 to 2.0, linear
    # Two-pass strategy: coarse grid first to find the posterior peak,
    # then refine around it.  The posterior is very narrow in z_s because
    # the data strongly constrain M * D_eff(z_s), so coarse spacing aliases
    # the peak into disconnected islands.
    # From the coarse run the posterior lives at z_s ~ 0.35-0.50,
    # M ~ 3e14 - 1.2e15; we zoom in with a fine grid there.
    mass_grid = np.logspace(13.8, 15.2, 120)   # finer, zoomed in
    zs_grid   = np.linspace(0.31, 0.80, 120)   # fine spacing, > z_halo=0.3
 
    log_post = compute_posterior_grid(gt_hsm, gt_err_hsm, mass_grid, zs_grid)
    np.save(OUTPUT_DIR / "q2d_log_posterior.npy", log_post)
 
    print("\nPlotting joint posterior...")
    mass_map, zs_map = plot_joint_posterior(
        log_post, mass_grid, zs_grid, OUTPUT_DIR
    )
 
    print("\nPlotting mass posterior at fixed z_s = 0.6...")
    plot_mass_posterior_fixed_zs(
        log_post, mass_grid, zs_grid, fixed_zs=0.6, output_dir=OUTPUT_DIR
    )
 
    print("\nPlotting MAP theory overlay...")
    # For the fixed-z_s overlay, find the best-fit mass at z_s=0.6 from
    # the posterior slice, not the joint MAP (which is at a different z_s)
    j_06       = np.argmin(np.abs(zs_grid - 0.6))
    mass_map_06 = mass_grid[np.argmax(log_post[:, j_06])]
    plot_best_fit_overlay(gt_hsm, gt_err_hsm, mass_map, zs_map,
                          mass_map_06, OUTPUT_DIR)
 
    print("\nQ2 complete.")
 
 
if __name__ == "__main__":
    main()

Loaded 100 halos and 100000 sources

Method: unweighted
  Shape noise variance sigma_e^2 = 0.04876
  Bin pair counts (simple): [   239    460    966   1947   4074   8284  16456  33407  66703 130574]

Method: weighted
  Shape noise variance sigma_e^2 = 0.00067
  Bin pair counts (simple): [   253    493   1047   2099   4354   8904  17723  35978  71728 140560]

Method: hsm
  Shape noise variance sigma_e^2 = 0.00328
  Bin pair counts (simple): [   253    493   1047   2099   4355   8905  17723  35979  71729 140562]

Plotting tangential shear profiles...
  Saved outputs/q2c_tangential_shear_profiles.png

Running posterior grid for HSM estimator...

Plotting joint posterior...
  MAP: mass = 7.627e+14 M_sun,  z_s = 0.364
  Saved outputs/q2d_joint_posterior.png

Plotting mass posterior at fixed z_s = 0.6...
  Fixed z_s = 0.598:
    MAP mass = 1.766e+14 M_sun
    68% CI   = [1.719e+14, 1.865e+14] M_sun
  Saved outputs/q2d_mass_posterior_fixed_zs.png

Plotting MAP theory overlay...
  Saved output